In [7]:
# K-Means ile Fizyolojik Profiller (Fenotipler) Çıkarılıyor...

# LEVEL-0: Hedef Dönüşümlü Kutsal Üçlü Eğitimi Başlıyor...

# --- FOLD 1 ---
#   XGBoost  RMSE : 1.22218
#   LightGBM RMSE : 1.23060
#   CatBoost RMSE : 1.22315
# --- FOLD 2 ---
#   XGBoost  RMSE : 1.22292
#   LightGBM RMSE : 1.22960
#   CatBoost RMSE : 1.21793
# --- FOLD 3 ---
#   XGBoost  RMSE : 1.20988
#   LightGBM RMSE : 1.21665
#   CatBoost RMSE : 1.20219
# --- FOLD 4 ---
#   XGBoost  RMSE : 1.21827
#   LightGBM RMSE : 1.22626
#   CatBoost RMSE : 1.21121
# --- FOLD 5 ---
#   XGBoost  RMSE : 1.23895
#   LightGBM RMSE : 1.24567
#   CatBoost RMSE : 1.23289

# LEVEL-1: Meta-Model Eğitimi (Tahminlerin Harmanlanması)...

# ----------------------------------------
# LEVEL-0 XGBoost  Ortalama RMSE : 1.22248
# LEVEL-0 LightGBM Ortalama RMSE : 1.22979
# LEVEL-0 CatBoost Ortalama RMSE : 1.21752
# ----------------------------------------
# 🌟 LEVEL-1 META-MODEL STACKING RMSE: 1.21656 🌟
# ----------------------------------------


import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, PowerTransformer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')

# ============================================================
# 1. VERİ YÜKLEME VE ÖZELLİK MÜHENDİSLİĞİ
# ============================================================
train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')
target = 'bilissel_performans_skoru'

def ozellik_uret(df):
    df = df.copy()
    df['toplam_kaliteli_uyku'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['uyku_bolunme_skoru'] = df['gecelik_uyanma_sayisi'] * (1 + df['uykuya_dalma_suresi_dk'] / 30)
    df['rem_derin_denge'] = df['rem_yuzdesi'] / (df['derin_uyku_yuzdesi'] + 1e-5) 
    df['sosyal_jetlag_kare'] = df['hafta_sonu_uyku_farki_saat'] ** 2
    df['uyku_sabotaj'] = (df['uyku_oncesi_kafein_mg'] * 0.5) + (df['uyku_oncesi_ekran_suresi_dk'] * 0.3)
    df['kafein_rem_baskisi'] = df['uyku_oncesi_kafein_mg'] * (df['rem_yuzdesi'] / 100)
    df['sicaklik_termal_stres'] = (df['oda_sicakligi_celsius'] - 23) ** 2

    gamma = 23.657
    theta = 4.260
    delta = 7.879
    df['bmi_etkisi'] = np.where(
        df['vucut_kitle_indeksi'] <= gamma,
        df['vucut_kitle_indeksi'] * theta,
        (gamma * theta) - ((df['vucut_kitle_indeksi'] - gamma) * delta)
    )

    df['adim_esik_ustu'] = (df['gunluk_adim_sayisi'] - 7500).clip(lower=0)
    df['aktivite_stres_tamponu'] = df['gunluk_adim_sayisi'] / (df['stres_skoru'] + 1)
    df['sekerleme_optimal_mi'] = ((df['sekerleme_suresi_dk'] >= 30) & (df['sekerleme_suresi_dk'] <= 60)).astype(int)
    df['sekerleme_uzamis_risk'] = (df['sekerleme_suresi_dk'] > 90).astype(int)
    df['nabiz_riski'] = (df['dinlenik_nabiz_bpm'] - 60).clip(lower=0)
    df['stres_calisma_yuku'] = df['stres_skoru'] * df['gunluk_calisma_saati']
    df['genel_stres_indeksi'] = (df['stres_skoru'] * 0.4) + (df['gecelik_uyanma_sayisi'] * 2.0) + \
                                (df['hafta_sonu_uyku_farki_saat'] * 0.5)
    
train = ozellik_uret(train)
test  = ozellik_uret(test)

X = train.drop(columns=[target, 'id'])
y = train[target]
test_ids = test['id']
X_test = test.drop(columns=['id'])

# ============================================================
# 2. K-MEANS FENOTİP (PROFİL) KÜMELEMESİ
# ============================================================
print("K-Means ile Fizyolojik Profiller (Fenotipler) Çıkarılıyor...")

# Kümeleme için kullanılacak fiziksel ve stres bazlı özellikler
cluster_cols = ['vucut_kitle_indeksi', 'dinlenik_nabiz_bpm', 'gunluk_adim_sayisi', 'stres_skoru']

# DİKKAT: KMeans ve StandardScaler NaN kabul etmez. 
# Bu yüzden ağaç modellerinin göreceği asıl veriyi bozmadan, 
# SADECE kümeleme için kullanılacak verideki eksikleri medyan ile dolduruyoruz.
imputer = SimpleImputer(strategy='median')
X_cluster_imputed = imputer.fit_transform(X[cluster_cols])
X_test_cluster_imputed = imputer.transform(X_test[cluster_cols])

# Uzaklık tabanlı algoritma olduğu için özellikleri aynı teraziye alıyoruz (Standartlaştırma)
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster_imputed)
X_test_cluster_scaled = scaler.transform(X_test_cluster_imputed)

# 4 Farklı Profil Çıkarıyoruz (Sağlıklı Hareketli, Stresli Sedanter vb.)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# Kümeleri bul ve asıl X ve X_test içine yeni bir özellik olarak ekle
X['fizyolojik_profil_id'] = kmeans.fit_predict(X_cluster_scaled)
X_test['fizyolojik_profil_id'] = kmeans.predict(X_test_cluster_scaled)
# ============================================================
# 3. KATEGORİK DEĞİŞKEN HAZIRLIĞI
# ============================================================
# Kategorik değişkenleri al ve yeni eklediğimiz Küme ID'sini de kategorik yap
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
cat_cols.append('fizyolojik_profil_id') # K-Means çıktısını kategorik olarak tanıtıyoruz

for col in cat_cols:
    X[col] = X[col].fillna('bilinmiyor').astype(str).astype('category')
    X_test[col] = X_test[col].fillna('bilinmiyor').astype(str).astype('category')

# ============================================================
# 4. K-FOLD STACKING & HEDEF DÖNÜŞÜMÜ (TARGET TRANSFORMATION)
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X))
oof_lgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

test_xgb = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

print("\nLEVEL-0: Hedef Dönüşümlü Kutsal Üçlü Eğitimi Başlıyor...\n")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"--- FOLD {fold+1} ---")
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # YENİ: Hedef Değişkeni Yeo-Johnson ile Normal Dağılıma (Çan Eğrisi) Çeviriyoruz
    # Bunu SADECE bu fold'un eğitim verisine (y_tr) bakarak fit ediyoruz (Sızıntıyı önlemek için)
    pt = PowerTransformer(method='yeo-johnson')
    y_tr_trans = pt.fit_transform(y_tr.values.reshape(-1, 1)).flatten()
    y_va_trans = pt.transform(y_va.values.reshape(-1, 1)).flatten()

    # ----------------------------------------------------
    # MODEL 1: XGBOOST (Dönüştürülmüş Hedef İle)
    # ----------------------------------------------------
    xgb_model = XGBRegressor(
        n_estimators=3000, learning_rate=0.015, max_depth=6,
        subsample=0.8, colsample_bytree=0.7, enable_categorical=True,
        tree_method='hist', early_stopping_rounds=150, random_state=42+fold, n_jobs=-1
    )
    # Eğitimi dönüştürülmüş Y ile yapıyoruz
    xgb_model.fit(X_tr, y_tr_trans, eval_set=[(X_va, y_va_trans)], verbose=False)
    
    # Tahminleri al ve "inverse_transform" ile GERÇEK puana geri çevir
    val_preds_trans = xgb_model.predict(X_va)
    oof_xgb[val_idx] = pt.inverse_transform(val_preds_trans.reshape(-1, 1)).flatten()
    
    # Test tahminlerini al ve GERÇEK puana çevirerek biriktir
    test_preds_trans = xgb_model.predict(X_test)
    test_xgb += pt.inverse_transform(test_preds_trans.reshape(-1, 1)).flatten() / kf.n_splits
    
    print(f"  XGBoost  RMSE : {np.sqrt(mean_squared_error(y_va, oof_xgb[val_idx])):.5f}")

    # ----------------------------------------------------
    # MODEL 2: LIGHTGBM
    # ----------------------------------------------------
    lgb_model = LGBMRegressor(
        n_estimators=3000, learning_rate=0.015, max_depth=6,
        subsample=0.8, colsample_bytree=0.7, random_state=42+fold, n_jobs=-1,
        verbose=-1 
    )
    lgb_model.fit(
        X_tr, y_tr_trans, 
        eval_set=[(X_va, y_va_trans)], 
        callbacks=[pd.core.common.import_optional_dependency("lightgbm").early_stopping(150, verbose=False)] 
        if hasattr(pd.core.common, 'import_optional_dependency') else None
    )
    oof_lgb[val_idx] = pt.inverse_transform(lgb_model.predict(X_va).reshape(-1, 1)).flatten()
    test_lgb += pt.inverse_transform(lgb_model.predict(X_test).reshape(-1, 1)).flatten() / kf.n_splits
    print(f"  LightGBM RMSE : {np.sqrt(mean_squared_error(y_va, oof_lgb[val_idx])):.5f}")

    # ----------------------------------------------------
    # MODEL 3: CATBOOST
    # ----------------------------------------------------
    cat_model = CatBoostRegressor(
        iterations=3000, learning_rate=0.015, depth=6,
        random_state=42+fold, early_stopping_rounds=150, 
        verbose=False, cat_features=cat_cols
    )
    cat_model.fit(X_tr, y_tr_trans, eval_set=(X_va, y_va_trans))
    oof_cat[val_idx] = pt.inverse_transform(cat_model.predict(X_va).reshape(-1, 1)).flatten()
    test_cat += pt.inverse_transform(cat_model.predict(X_test).reshape(-1, 1)).flatten() / kf.n_splits
    print(f"  CatBoost RMSE : {np.sqrt(mean_squared_error(y_va, oof_cat[val_idx])):.5f}")

# ============================================================
# 5. LEVEL-1 MİMARİSİ: META-MODEL (STACKING)
# ============================================================
print("\nLEVEL-1: Meta-Model Eğitimi (Tahminlerin Harmanlanması)...\n")

X_meta_train = np.column_stack([oof_xgb, oof_lgb, oof_cat])
X_meta_test = np.column_stack([test_xgb, test_lgb, test_cat])

# Meta modeli, modellerden gelen GERÇEK puanlar ile orijinal Y üzerinden eğitiyoruz
meta_model = Ridge(alpha=1.0)
meta_model.fit(X_meta_train, y)

final_oof_preds = meta_model.predict(X_meta_train)
genel_rmse = np.sqrt(mean_squared_error(y, final_oof_preds))

print("-" * 40)
print(f"LEVEL-0 XGBoost  Ortalama RMSE : {np.sqrt(mean_squared_error(y, oof_xgb)):.5f}")
print(f"LEVEL-0 LightGBM Ortalama RMSE : {np.sqrt(mean_squared_error(y, oof_lgb)):.5f}")
print(f"LEVEL-0 CatBoost Ortalama RMSE : {np.sqrt(mean_squared_error(y, oof_cat)):.5f}")
print("-" * 40)
print(f"🌟 LEVEL-1 META-MODEL STACKING RMSE: {genel_rmse:.5f} 🌟")
print("-" * 40)

# ============================================================
# GÖNDERİM
# ============================================================
final_test_preds = meta_model.predict(X_meta_test)
submission = pd.DataFrame({'id': test_ids, target: final_test_preds})
submission.to_csv('submission.csv', index=False)
print("\nGelişmiş Stacking Submission 'submission.csv' olarak başarıyla oluşturuldu!")

K-Means ile Fizyolojik Profiller (Fenotipler) Çıkarılıyor...

LEVEL-0: Hedef Dönüşümlü Kutsal Üçlü Eğitimi Başlıyor...

--- FOLD 1 ---
  XGBoost  RMSE : 1.22321
  LightGBM RMSE : 1.23126
  CatBoost RMSE : 1.22390
--- FOLD 2 ---
  XGBoost  RMSE : 1.22229
  LightGBM RMSE : 1.23062
  CatBoost RMSE : 1.21821
--- FOLD 3 ---
  XGBoost  RMSE : 1.20942
  LightGBM RMSE : 1.21627
  CatBoost RMSE : 1.20245
--- FOLD 4 ---
  XGBoost  RMSE : 1.21830
  LightGBM RMSE : 1.22539
  CatBoost RMSE : 1.21119
--- FOLD 5 ---
  XGBoost  RMSE : 1.23754
  LightGBM RMSE : 1.24525
  CatBoost RMSE : 1.23278

LEVEL-1: Meta-Model Eğitimi (Tahminlerin Harmanlanması)...

----------------------------------------
LEVEL-0 XGBoost  Ortalama RMSE : 1.22218
LEVEL-0 LightGBM Ortalama RMSE : 1.22979
LEVEL-0 CatBoost Ortalama RMSE : 1.21775
----------------------------------------
🌟 LEVEL-1 META-MODEL STACKING RMSE: 1.21662 🌟
----------------------------------------

Gelişmiş Stacking Submission 'submission.csv' olarak başarıyl